In [6]:
from pathlib import Path
import pandas as pd

# Resolve project root cleanly whether notebook runs from ./Scripts or root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Scripts" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DIR = PROJECT_ROOT / "Processed"
PROCESSED_DIR.mkdir(exist_ok=True)

# Load compressed matrices directly (tsv.gz uses tab, csv.gz uses comma)
df_circ = pd.read_csv(DATA_DIR / "GSE236932_circ_countmatrix.tsv.gz", sep="\t")
df_mrna = pd.read_csv(DATA_DIR / "GSE236932_mRNA_countmatrix.csv.gz")

print(f"circRNA shape: {df_circ.shape}")
print("circRNA first 10 columns:", df_circ.columns[:10].tolist())
print("circRNA last 5 columns:", df_circ.columns[-5:].tolist())
print(df_circ.iloc[:2, :8])

print(f"\nmRNA shape: {df_mrna.shape}")
print("mRNA first 10 columns:", df_mrna.columns[:10].tolist())
print("mRNA last 5 columns:", df_mrna.columns[-5:].tolist())
print(df_mrna.iloc[:2, :8])

circRNA shape: (16112, 63)
circRNA first 10 columns: ['258510BT_FRRL210032199-1a', '263772BT_FRRL210024289-1a', '264495BN_FRRL210032210-1a', '264495BT_FRRL210024288-1a', '265902BP_FRRL210032215-1a', '265902BT_FRRL210032217-1a', '273879BP_FRRL202251962-1a', '273879BT_FRRL202251961-1a', '273902BN_FRRL210032186-1a', '273902BT_FRRL202251959-1a']
circRNA last 5 columns: ['B300069BP', 'B300069BT', 'B300180BT', 'B300378BP', 'B300378BT']
                                    258510BT_FRRL210032199-1a  \
chr1_KI270711v1_random:20007|25239                          2   
chr1:100007034|100017815                                    0   

                                    263772BT_FRRL210024289-1a  \
chr1_KI270711v1_random:20007|25239                          8   
chr1:100007034|100017815                                    0   

                                    264495BN_FRRL210032210-1a  \
chr1_KI270711v1_random:20007|25239                          0   
chr1:100007034|100017815                    

In [5]:
df_circ

,258510BT_FRRL210032199-1a,263772BT_FRRL210024289-1a,264495BN_FRRL210032210-1a,264495BT_FRRL210024288-1a,265902BP_FRRL210032215-1a,265902BT_FRRL210032217-1a,273879BP_FRRL202251962-1a,273879BT_FRRL202251961-1a,273902BN_FRRL210032186-1a,273902BT_FRRL202251959-1a,...,B297969BT,B297993BT,B298079BT,B298182BT,B299391BT,B300069BP,B300069BT,B300180BT,B300378BP,B300378BT
chr1_KI270711v1_random:20007|25239,2,8,0,0,13,0,14,19,23,0,...,0,0,0,0,0,0,0,7,0,0
chr1:100007034|100017815,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,2,0,0,0
chr1:100011365|100017815,0,0,0,0,0,0,0,0,0,0,...,0,5,0,0,0,0,0,0,0,2
chr1:100043073|100061949,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chr1:100049909|100061949,0,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrY:5030899|5100707,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chrY:7341115|7356230,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chrY:7341115|7371889,0,0,9,4,0,0,0,0,8,5,...,0,0,0,0,0,0,0,3,0,0
chrY:8068907|8070117,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,4,0,0


In [ ]:
import json
import numpy as np
from scipy.stats import zscore, f_oneway, ttest_ind

# 1. Verify and align the exact 63 sample columns between circRNA and mRNA
mrna_sample_cols = [c for c in df_mrna.columns if c not in ['genename', 'geneID']]
circ_sample_cols = df_circ.columns.tolist()

# Check that both matrices have the exact same 63 samples
assert set(circ_sample_cols) == set(mrna_sample_cols), "Sample IDs do not match!"

# Parse tissue group ('BT' = Tumor, 'BP' = Para-carcinoma, 'BN' = Normal) from sample IDs
def get_tissue_group(sample_id):
    prefix = sample_id.split('_')[0]  # e.g., '258510BT' or 'B300069BP'
    if prefix.endswith('BT'):
        return 'Tumor'
    elif prefix.endswith('BP'):
        return 'Para_Carcinoma'
    elif prefix.endswith('BN'):
        return 'Normal'
    return 'Unknown'

sample_meta = pd.DataFrame({
    'Sample_ID': circ_sample_cols,
    'Group': [get_tissue_group(s) for s in circ_sample_cols]
}).set_index('Sample_ID')

# Sort samples cleanly by Group (Tumor -> Para_Carcinoma -> Normal) and apply order to both matrices
sample_meta = sample_meta.sort_values(by=['Group', 'Sample_ID'], ascending=[False, True])
ordered_samples = sample_meta.index.tolist()

print("=== Sample Breakdown in GSE236932 ===")
print(sample_meta['Group'].value_counts())

# 2. Normalize circRNA raw counts -> log2(CPM + 1)
circ_counts = df_circ[ordered_samples].astype(float)
circ_lib_sizes = circ_counts.sum(axis=0)
X_circ_logcpm = np.log2(circ_counts.divide(circ_lib_sizes, axis=1) * 1e6 + 1)
X_circ_logcpm.index.name = 'circRNA_id'

# 3. Aggregate duplicate gene symbols in mRNA & log2(x + 1) transform (already FPKM/TPM)
df_mrna_clean = df_mrna.dropna(subset=['genename']).groupby('genename')[ordered_samples].sum()
mrna_log2 = np.log2(df_mrna_clean.astype(float) + 1)

# 4. Compute continuous target scores (Y) across all 63 samples
signatures = {
    "Hypoxia_Buffa": [
        "VEGFA", "SLC2A1", "PGAM1", "ENO1", "LDHA", "TPI1", "ALDOA", 
        "PGK1", "GPI", "P4HA1", "MRPS17", "CDKN3", "ADM", "NDRG1", "TUBB6"
    ],
    "EMT_Invasion": [
        "VIM", "CDH2", "FN1", "SNAI1", "SNAI2", "TWIST1", "ZEB1", 
        "ZEB2", "MMP2", "MMP9", "COL1A1", "COL3A1", "TGFB1", "SPARC"
    ],
    "Immune_Exhaustion_TME": [
        "CD274", "PDCD1", "CTLA4", "LAG3", "HAVCR2", "TIGIT", 
        "IDO1", "TGFB1", "IL10", "FOXP3", "CCL2", "CXCL8"
    ]
}

scores_df = sample_meta.copy()

print("\n=== Signature Separation Across 63 Bladder Samples ===")
tumor_ids = scores_df[scores_df['Group'] == 'Tumor'].index
normal_ids = scores_df[scores_df['Group'] == 'Normal'].index

for sig_name, gene_list in signatures.items():
    present_genes = [g for g in gene_list if g in mrna_log2.index]
    sig_matrix = mrna_log2.loc[present_genes]
    sig_matrix = sig_matrix.loc[sig_matrix.std(axis=1) > 0]

    # Row-wise Z-score across all 63 samples, then average per sample
    sig_z = sig_matrix.apply(zscore, axis=1)
    scores_df[sig_name] = sig_z.mean(axis=0)

    t_stat, p_val = ttest_ind(scores_df.loc[tumor_ids, sig_name], 
                              scores_df.loc[normal_ids, sig_name])
    print(f"{sig_name} ({len(present_genes)}/{len(gene_list)} genes): "
          f"Tumor vs Normal t={t_stat:.2f}, p={p_val:.4e}")

print("\n=== Mean Signature Score by Tissue Group ===")
print(scores_df.groupby('Group')[list(signatures.keys())].mean().round(3))

# 5. Save processed artifacts for Notebook 02
X_circ_logcpm.to_csv(PROCESSED_DIR / "GSE236932_circRNA_log2cpm.csv")
mrna_log2.to_csv(PROCESSED_DIR / "GSE236932_mRNA_log2.csv")
scores_df.to_csv(PROCESSED_DIR / "GSE236932_target_scores.csv")

with open(PROCESSED_DIR / "signatures.json", "w") as f:
    json.dump(signatures, f, indent=2)

print("\nAll processed matrices saved successfully to:", PROCESSED_DIR)

=== Sample Breakdown in GSE236932 ===
Group
Tumor             38
Para_Carcinoma    15
Normal            10
Name: count, dtype: int64

=== Signature Separation Across 63 Bladder Samples ===
Hypoxia_Buffa (15/15 genes): Tumor vs Normal t=3.93, p=2.8152e-04
EMT_Invasion (14/14 genes): Tumor vs Normal t=-3.41, p=1.3605e-03
Immune_Exhaustion_TME (12/12 genes): Tumor vs Normal t=-1.57, p=1.2352e-01

=== Mean Signature Score by Tissue Group ===
                Hypoxia_Buffa  EMT_Invasion  Immune_Exhaustion_TME
Group                                                             
Normal                 -0.480         0.497                  0.144
Para_Carcinoma         -0.250         0.224                  0.214
Tumor                   0.225        -0.219                 -0.123

All processed matrices saved successfully to: /Users/hahn/Library/CloudStorage/OneDrive-Personal/Birmingham-PhD/GSE236932-circDriver/Processed
